# 7 — Tuning Ridge & Lasso 
**Khusus training Januari–April**: gunakan `TimeSeriesSplit` expanding window pada **jam unik**, sehingga zona pada timestamp yang sama tidak terpecah train/validasi. Mei tetap validation akhir, Juni tidak disentuh. Tuning ini **tidak menimpa** dataset fitur atau output Fase 4A.

Jalankan setelah `06_regression.ipynb` (notebook ini mandiri, tidak bergantung pada variabel notebook lain).

In [2]:
from pathlib import Path
import sys,time
import numpy as np,pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import Ridge,Lasso
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
ROOT=Path.cwd().resolve()
if not (ROOT/'datasets').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'script'/'tools'))
import metrics
DATA=ROOT/'datasets/interim/2026/features_dataset.parquet'
OUT=ROOT/'out/regression';OUT.mkdir(parents=True,exist_ok=True)
NUMERIC=['hour_of_day','day_of_week','month','is_weekend','is_holiday','is_major_nyc_event',
'PU_longitude','PU_latitude','is_airport','is_cbd','lag_1','lag_24','lag_168','roll_24','roll_168']
FEATURES=NUMERIC+['PU_borough']
ALPHAS_RIDGE=[0.1,1,10,100,1000]
ALPHAS_LASSO=[0.001,0.01,0.1,1.0]
N_SPLITS=3
MIN_TRAIN_DAYS=21
SEED=42


In [3]:
df=pd.read_parquet(DATA)
df['hour']=pd.to_datetime(df['hour'])
train=df[df.split=='train'].sort_values(['hour','zone_id']).reset_index(drop=True)
val=df[df.split=='validation'].sort_values(['hour','zone_id']).reset_index(drop=True)
assert len(train)>0 and len(val)>0
assert train.hour.max()<val.hour.min()
assert train[FEATURES+['target_pickup_count','lag_168']].notna().all().all()
assert val[FEATURES+['target_pickup_count']].notna().all().all()
unique_hours=np.array(sorted(train.hour.unique()))
# validation blok waktu harus lebih besar daripada satu hari; expanding origin
folds=TimeSeriesSplit(n_splits=N_SPLITS).split(unique_hours)
fold_specs=[]
for train_idx,valid_idx in folds:
    a=unique_hours[train_idx];b=unique_hours[valid_idx]
    assert len(a)>=MIN_TRAIN_DAYS*24, 'Fold terlalu pendek; kurangi n_splits atau MIN_TRAIN_DAYS'
    assert a.max()<b.min()
    fold_specs.append((a.min(),a.max(),b.min(),b.max()))
print('Fold waktu:',*fold_specs,sep='\n')


Fold waktu:
(Timestamp('2026-01-08 00:00:00'), Timestamp('2026-02-03 13:00:00'), Timestamp('2026-02-03 14:00:00'), Timestamp('2026-03-02 00:00:00'))
(Timestamp('2026-01-08 00:00:00'), Timestamp('2026-03-02 00:00:00'), Timestamp('2026-03-02 01:00:00'), Timestamp('2026-04-04 12:00:00'))
(Timestamp('2026-01-08 00:00:00'), Timestamp('2026-04-04 12:00:00'), Timestamp('2026-04-04 13:00:00'), Timestamp('2026-04-30 23:00:00'))


In [4]:
def model_for(kind,alpha):
    pre=ColumnTransformer([('numeric',StandardScaler(),NUMERIC),('borough',OneHotEncoder(handle_unknown='ignore'),['PU_borough'])],sparse_threshold=0)
    reg=Ridge(alpha=alpha,random_state=SEED) if kind=='Ridge' else Lasso(alpha=alpha,max_iter=2500,tol=1e-3)
    return Pipeline([('pre',pre),('reg',reg)])

rows=[]
for kind,alphas in [('Ridge',ALPHAS_RIDGE),('Lasso',ALPHAS_LASSO)]:
    for alpha in alphas:
        for fold,(train_start,train_end,val_start,val_end) in enumerate(fold_specs,1):
            tr=train[(train.hour>=train_start)&(train.hour<=train_end)]
            va=train[(train.hour>=val_start)&(train.hour<=val_end)]
            model=model_for(kind,alpha)
            begin=time.perf_counter();model.fit(tr[FEATURES],tr.target_pickup_count)
            train_seconds=time.perf_counter()-begin
            begin=time.perf_counter();pred=np.maximum(model.predict(va[FEATURES]),0)
            pred_seconds=time.perf_counter()-begin
            rows.append({'model':kind,'alpha':alpha,'fold':fold,'mae':mean_absolute_error(va.target_pickup_count,pred),
                         'train_seconds':train_seconds,'predict_seconds':pred_seconds,
                         'train_end':str(train_end),'validation_start':str(val_start)})
            print(kind,alpha,'fold',fold,'MAE',round(rows[-1]['mae'],3))
cv=pd.DataFrame(rows)
cv.to_csv(OUT/'07_tuning_fold_results.csv',index=False)
summary=cv.groupby(['model','alpha'],as_index=False).agg(cv_mae_mean=('mae','mean'),cv_mae_std=('mae','std'),
training_seconds=('train_seconds','sum'),prediction_seconds=('predict_seconds','sum'))
summary=summary.sort_values(['model','cv_mae_mean'])
display(summary.round(4))
summary.to_csv(OUT/'07_tuning_summary.csv',index=False)


Ridge 0.1 fold 1 MAE 5.677
Ridge 0.1 fold 2 MAE 4.156
Ridge 0.1 fold 3 MAE 4.057
Ridge 1 fold 1 MAE 5.677
Ridge 1 fold 2 MAE 4.156
Ridge 1 fold 3 MAE 4.057
Ridge 10 fold 1 MAE 5.677
Ridge 10 fold 2 MAE 4.156
Ridge 10 fold 3 MAE 4.057
Ridge 100 fold 1 MAE 5.674
Ridge 100 fold 2 MAE 4.155
Ridge 100 fold 3 MAE 4.057
Ridge 1000 fold 1 MAE 5.656
Ridge 1000 fold 2 MAE 4.145
Ridge 1000 fold 3 MAE 4.052


C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.622095e+06, tolerance: 4.755e+05
  model = cd_fast.enet_coordinate_descent(


Lasso 0.001 fold 1 MAE 5.671


C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.092867e+07, tolerance: 9.430e+05
  model = cd_fast.enet_coordinate_descent(


Lasso 0.001 fold 2 MAE 4.156


C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.636559e+07, tolerance: 1.434e+06
  model = cd_fast.enet_coordinate_descent(


Lasso 0.001 fold 3 MAE 4.057
Lasso 0.01 fold 1 MAE 5.628
Lasso 0.01 fold 2 MAE 4.154
Lasso 0.01 fold 3 MAE 4.054
Lasso 0.1 fold 1 MAE 5.197
Lasso 0.1 fold 2 MAE 4.147
Lasso 0.1 fold 3 MAE 4.041
Lasso 1.0 fold 1 MAE 4.344
Lasso 1.0 fold 2 MAE 4.194
Lasso 1.0 fold 3 MAE 4.138


,model,alpha,cv_mae_mean,cv_mae_std,training_seconds,prediction_seconds
3,Lasso,1.000,4.2255,0.1068,2.7147,0.2843
2,Lasso,0.100,4.4617,0.6390,5.3838,0.3468
1,Lasso,0.010,4.6118,0.8812,35.9084,0.3030
0,Lasso,0.001,4.6281,0.9049,171.5979,0.4177
8,Ridge,1000.000,4.6178,0.9006,1.6218,0.4096
7,Ridge,100.000,4.6287,0.9070,1.8252,0.4584
6,Ridge,10.000,4.6300,0.9079,1.4910,0.3415
5,Ridge,1.000,4.6301,0.9080,1.5616,0.3529
4,Ridge,0.100,4.6302,0.9080,1.9785,0.3986


In [5]:
# Pemilihan alpha berdasarkan CV TRAIN saja; evaluasi satu kali pada Mei.
best=summary.sort_values('cv_mae_mean').groupby('model',sort=False).first().reset_index()
rows=[]
scale=metrics.mase_scale(train.target_pickup_count,train.lag_168)
for r in best.itertuples():
    model=model_for(r.model,r.alpha)
    t=time.perf_counter();model.fit(train[FEATURES],train.target_pickup_count);training_s=time.perf_counter()-t
    t=time.perf_counter();pred=np.maximum(model.predict(val[FEATURES]),0);pred_s=time.perf_counter()-t
    mm=metrics.regression_metrics(val.target_pickup_count,pred,scale=scale)
    rows.append({'model':r.model,'alpha':r.alpha,'cv_mae_mean':r.cv_mae_mean,'split':'validation',**mm,
                 'train_seconds':training_s,'predict_seconds':pred_s})
report=pd.DataFrame(rows).sort_values('mae')
report.to_csv(OUT/'07_tuning_validation_results.csv',index=False)
display(report.round(4))
print('Serahkan tiga file CSV tuning ke Anggota 4. TEST Juni belum disentuh.')


,model,alpha,cv_mae_mean,split,mae,rmse,wape,r2,mase,train_seconds,predict_seconds
1,Ridge,1000.0,4.6178,validation,4.1608,11.6007,0.2036,0.9584,0.8191,0.9893,0.1161
0,Lasso,1.0,4.2255,validation,4.2395,11.6552,0.2074,0.9580,0.8346,1.7599,0.1878


Serahkan tiga file CSV tuning ke Anggota 4. TEST Juni belum disentuh.
